In [1]:
import os
from pyspark.sql import SparkSession
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages graphframes:graphframes:0.8.4-spark3.5-s_2.12 pyspark-shell"
spark = SparkSession.builder.appName("Amanda_code").getOrCreate()
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from graphframes import GraphFrame
from pyspark.sql.functions import col, to_timestamp
from pyspark.sql.types import IntegerType, FloatType, BooleanType

df_raw = spark.read.csv("US_Accidents_March23.csv", header = True)
df_raw.printSchema()

root
 |-- ID: string (nullable = true)
 |-- Source: string (nullable = true)
 |-- Severity: string (nullable = true)
 |-- Start_Time: string (nullable = true)
 |-- End_Time: string (nullable = true)
 |-- Start_Lat: string (nullable = true)
 |-- Start_Lng: string (nullable = true)
 |-- End_Lat: string (nullable = true)
 |-- End_Lng: string (nullable = true)
 |-- Distance(mi): string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: string (nullable = true)
 |-- Temperature(F): string (nullable = true)
 |-- Wind_Chill(F): string (nullable = true)
 |-- Humidity(%): string (nullable = true)
 |-- Pressure(in): string (nullable = true)
 |-- Visibility(

In [2]:
df_raw.filter(col("Temperature(F)") < -56.0).count()

22

In [3]:
df_raw.filter(col("Temperature(F)") > 122.0).count()

56

In [4]:
df_raw.filter(
    col("Temperature(F)").isNull() &
    col("Weather_Condition").isNotNull() &
    col("Visibility(mi)").isNotNull()
).count()

25801

In [5]:
df_raw.filter(
    col("Temperature(F)").isNotNull() &
    col("Weather_Condition").isNull() &
    col("Visibility(mi)").isNotNull()
).count()

16458

In [6]:
df_raw.filter(
    col("Temperature(F)").isNotNull() &
    col("Weather_Condition").isNotNull() &
    col("Visibility(mi)").isNull()
).count()

19389

In [7]:
df_raw.filter(
    col("Temperature(F)").isNull() &
    col("Weather_Condition").isNull() &
    col("Visibility(mi)").isNotNull()
).count()

513

In [8]:
df_raw.filter(
    col("Temperature(F)").isNull() &
    col("Weather_Condition").isNotNull() &
    col("Visibility(mi)").isNull()
).count()

1221

In [9]:
df_raw.filter(
    col("Temperature(F)").isNotNull() &
    col("Weather_Condition").isNull() &
    col("Visibility(mi)").isNull()
).count()

20170

In [10]:
# SSS
df_raw.filter(
    col("Temperature(F)").isNull() &
    col("Weather_Condition").isNull() &
    col("Visibility(mi)").isNull()
).count()

136318

In [11]:
df = df_raw \
    .withColumn("Severity", col("Severity").cast(IntegerType())) \
    .withColumn("Start_Time", to_timestamp("Start_Time")) \
    .withColumn("End_Time", to_timestamp("End_Time")) \
    .withColumn("Weather_Timestamp", to_timestamp("Weather_Timestamp")) \
    .withColumn("Start_Lat", col("Start_Lat").cast(FloatType())) \
    .withColumn("Start_Lng", col("Start_Lng").cast(FloatType())) \
    .withColumn("End_Lat", col("End_Lat").cast(FloatType())) \
    .withColumn("End_Lng", col("End_Lng").cast(FloatType())) \
    .withColumn("Distance(mi)", col("Distance(mi)").cast(FloatType()))


boolean_cols = [
    "Amenity", "Bump", "Crossing", "Give_Way", "Junction",
    "No_Exit", "Railway", "Roundabout", "Station", "Stop",
    "Traffic_Calming", "Traffic_Signal", "Turning_Loop"
]

for c in boolean_cols:
    df = df.withColumn(c, col(c).cast(BooleanType()))

from pyspark.sql.functions import col
from pyspark.sql.types import FloatType

weather_float_cols = [
    "Temperature(F)",
    "Wind_Chill(F)",
    "Humidity(%)",
    "Pressure(in)",
    "Visibility(mi)",
    "Wind_Speed(mph)",
    "Precipitation(in)"
]

for c in weather_float_cols:
    df = df.withColumn(c, col(c).cast(FloatType()))

df.printSchema()

root
 |-- ID: string (nullable = true)
 |-- Source: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: float (nullable = true)
 |-- Start_Lng: float (nullable = true)
 |-- End_Lat: float (nullable = true)
 |-- End_Lng: float (nullable = true)
 |-- Distance(mi): float (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: timestamp (nullable = true)
 |-- Temperature(F): float (nullable = true)
 |-- Wind_Chill(F): float (nullable = true)
 |-- Humidity(%): float (nullable = true)
 |-- Pressure(in): float (nullable = true)
 |-- Visibility

In [12]:
from pyspark.sql.functions import col, sum

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts.show(vertical=True)

-RECORD 0------------------------
 ID                    | 0       
 Source                | 0       
 Severity              | 0       
 Start_Time            | 0       
 End_Time              | 0       
 Start_Lat             | 0       
 Start_Lng             | 0       
 End_Lat               | 3402762 
 End_Lng               | 3402762 
 Distance(mi)          | 0       
 Description           | 5       
 Street                | 10869   
 City                  | 253     
 County                | 0       
 State                 | 0       
 Zipcode               | 1915    
 Country               | 0       
 Timezone              | 7808    
 Airport_Code          | 22635   
 Weather_Timestamp     | 120228  
 Temperature(F)        | 163853  
 Wind_Chill(F)         | 1999019 
 Humidity(%)           | 174144  
 Pressure(in)          | 140679  
 Visibility(mi)        | 177098  
 Wind_Direction        | 175206  
 Wind_Speed(mph)       | 571233  
 Precipitation(in)     | 2203586 
 Weather_Condi

In [13]:
null_counts.count()


1

In [14]:

df.selectExpr(
    "min(`Temperature(F)`) as min_temp",
    "max(`Temperature(F)`) as max_temp",
    "min(`Visibility(mi)`) as min_vis",
    "max(`Visibility(mi)`) as max_vis"
).show()


+--------+--------+-------+-------+
|min_temp|max_temp|min_vis|max_vis|
+--------+--------+-------+-------+
|   -89.0|   207.0|    0.0|  140.0|
+--------+--------+-------+-------+



In [15]:
from pyspark.sql.functions import col

inconsistent_time = df.filter(
    col("End_Time").isNotNull() &
    col("Start_Time").isNotNull() &
    (col("End_Time") < col("Start_Time"))
)

inconsistent_time_count = inconsistent_time.count()
print(f"Inconsistent time records: {inconsistent_time_count}")


Inconsistent time records: 0


In [16]:
wind_chill_issues = df.filter(
    col("Wind_Chill(F)").isNotNull() &
    col("Temperature(F)").isNotNull() &
    (col("Wind_Chill(F)") > col("Temperature(F)"))
)

print(f"Wind chill > temperature records: {wind_chill_issues.count()}")

Wind chill > temperature records: 0


In [17]:
invalid_coordinates = df.filter(
    (col("Start_Lat") < -90) | (col("Start_Lat") > 90) |
    (col("Start_Lng") < -180) | (col("Start_Lng") > 180)
)

print(f"Invalid coordinate records: {invalid_coordinates.count()}")

Invalid coordinate records: 0


In [19]:
def iqr_outliers(df, column):
    q1, q3 = df.approxQuantile(column, [0.25, 0.75], 0.01)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return df.filter(
        (col(column) < lower) | (col(column) > upper)
    ), lower, upper

In [20]:
distance_outliers, lower_d, upper_d = iqr_outliers(df, "Distance(mi)")

print(f"Distance outliers count: {distance_outliers.count()}")
print(f"Accepted range: {lower_d:.2f} – {upper_d:.2f} miles")

Distance outliers count: 969428
Accepted range: -0.69 – 1.15 miles


In [21]:
temp_outliers, min_t, max_t = iqr_outliers(df, "Temperature(F)")

print(f"Temperature outliers count: {temp_outliers.count()}")
print(f"Accepted range: {min_t:.2f} – {max_t:.2f} °F")

Temperature outliers count: 50515
Accepted range: 8.50 – 116.50 °F


In [22]:
physical_violations = df.filter(
    (col("Temperature(F)") < -56) |
    (col("Temperature(F)") > 122) |
    (col("Visibility(mi)") < 0) |
    (col("Wind_Speed(mph)") < 0)
)

print(f"Physically impossible weather records: {physical_violations.count()}")

Physically impossible weather records: 78


In [23]:
from pyspark.sql.functions import year

accidents_per_year = df.withColumn(
    "Year", year("Start_Time")
).groupBy("Year").count().orderBy("Year")

accidents_per_year.show()

+----+-------+
|Year|  count|
+----+-------+
|2016| 410821|
|2017| 718093|
|2018| 893426|
|2019| 954303|
|2020|1178913|
|2021|1563753|
|2022|1762452|
|2023| 246633|
+----+-------+



In [24]:
from pyspark.sql.functions import month

accidents_per_month = df.withColumn(
    "Month", month("Start_Time")
).groupBy("Month").count().orderBy("Month")

accidents_per_month.show()

+-----+------+
|Month| count|
+-----+------+
|    1|751946|
|    2|658984|
|    3|554595|
|    4|587300|
|    5|558176|
|    6|571373|
|    7|512335|
|    8|599666|
|    9|651381|
|   10|675130|
|   11|760165|
|   12|847343|
+-----+------+



In [25]:
from pyspark.sql.functions import to_date

daily_counts = df.withColumn(
    "Date", to_date("Start_Time")
).groupBy("Date").count()

zero_days = daily_counts.filter(col("count") == 0)
print(f"Days with zero accidents recorded: {zero_days.count()}")

Days with zero accidents recorded: 0
